<a href="https://colab.research.google.com/github/Buddhiimz/DeepLearning_Project/blob/feature%2Fjithma/IT22095176_DL_Assigment_EfficientNetB0_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Step 1: Imports
import os, glob
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from sklearn.utils import class_weight
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
# Mount Google Drive to access datasets
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Extract train zip folders
import zipfile
with zipfile.ZipFile('/content/drive/MyDrive/Colab Notebooks/Assignment/train.zip', 'r') as zip_ref:
    zip_ref.extractall('train')

In [ ]:
# Extract test zip folders
import zipfile
with zipfile.ZipFile('/content/drive/MyDrive/Colab Notebooks/Assignment/test.zip', 'r') as zip_ref:
    zip_ref.extractall('test')

In [ ]:
!pip install -q tensorflow scikit-learn matplotlib

In [ ]:
# Step 1: User params

DATA_DIR = "/content/train/train"
IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 123
INITIAL_EPOCHS = 8
FINE_TUNE_EPOCHS = 25
FINE_TUNE_UNFREEZE_LAST_N = 40
BASE_LR = 1e-3
FT_LR = 1e-5
LABEL_SMOOTHING = 0.1
PATIENCE_ES = 8
PATIENCE_RLR = 3
MODEL_CHECKPOINT_PATH = "best_effnetb0.weights.h5"
USE_TTA = True


In [ ]:
# Step 2: Helper functions

def _has_class_folders_with_images(path):
    if not path or not os.path.isdir(path):
        return False
    for sub in os.listdir(path):
        subp = os.path.join(path, sub)
        if not os.path.isdir(subp):
            continue
        imgs = glob.glob(os.path.join(subp, "*.jpg")) + \
               glob.glob(os.path.join(subp, "*.jpeg")) + \
               glob.glob(os.path.join(subp, "*.png"))
        if imgs:
            return True
    return False

In [ ]:
# Step 3: Set data directories

data_dir = DATA_DIR
val_dir = "/content/test/test"
if not _has_class_folders_with_images(data_dir):
    raise FileNotFoundError(f"No images found in {data_dir}. Ensure it contains class subfolders with images.")
# Optional: if you have a separate validation folder inside Assignment
val_dir = os.path.join(data_dir, "val")
if not _has_class_folders_with_images(val_dir):
    val_dir = None

print("Using data_dir =", data_dir)
if val_dir:
    print("Using val_dir =", val_dir)



Using data_dir = /content/train/train


In [ ]:
# Step 4: Create datasets

AUTOTUNE = tf.data.AUTOTUNE

if val_dir:
    train_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir, seed=SEED, image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE, label_mode='categorical', shuffle=True)
    val_ds = tf.keras.utils.image_dataset_from_directory(
        val_dir, seed=SEED, image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE, label_mode='categorical', shuffle=False)
else:
    train_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir, validation_split=0.2, subset="training", seed=SEED,
        image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE, label_mode='categorical', shuffle=True)
    val_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir, validation_split=0.2, subset="validation", seed=SEED,
        image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE, label_mode='categorical', shuffle=False)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print("Discovered classes:", class_names)

train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

Found 2820 files belonging to 52 classes.
Using 2256 files for training.
Found 2820 files belonging to 52 classes.
Using 564 files for validation.
Discovered classes: ['n0000', 'n0001', 'n0002', 'n0003', 'n0004', 'n0005', 'n0006', 'n0007', 'n0008', 'n0009', 'n0010', 'n0011', 'n0012', 'n0013', 'n0014', 'n0015', 'n0016', 'n0017', 'n0018', 'n0019', 'n0020', 'n0021', 'n0022', 'n0023', 'n0024', 'n0025', 'n0026', 'n0027', 'n0028', 'n0029', 'n0030', 'n0031', 'n0032', 'n0033', 'n0034', 'n0035', 'n0036', 'n0037', 'n0038', 'n0039', 'n0040', 'n0041', 'n0042', 'n0043', 'n0044', 'n0045', 'n0046', 'n0047', 'n0048', 'n0049', 'n0050', 'n0051']


In [ ]:
# Step 5: Compute class weights

y_all = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in train_ds], axis=0)
cw = class_weight.compute_class_weight(class_weight='balanced', classes=np.unique(y_all), y=y_all)
class_weights = {i: float(v) for i, v in enumerate(cw)}
print("Computed class weights:", class_weights)

Computed class weights: {0: 1.2395604395604396, 1: 1.1124260355029585, 2: 2.7115384615384617, 3: 1.032967032967033, 4: 2.7115384615384617, 5: 1.8076923076923077, 6: 2.41025641025641, 7: 3.337278106508876, 8: 2.283400809716599, 9: 1.1725571725571726, 10: 3.337278106508876, 11: 0.6380090497737556, 12: 1.1417004048582995, 13: 0.7888111888111888, 14: 1.0581613508442778, 15: 1.2760180995475112, 16: 0.8034188034188035, 17: 0.36766623207301175, 18: 0.5634365634365635, 19: 0.9431438127090301, 20: 0.5423076923076923, 21: 2.169230769230769, 22: 2.41025641025641, 23: 2.7115384615384617, 24: 0.3213675213675214, 25: 2.7115384615384617, 26: 1.7353846153846153, 27: 0.6197802197802198, 28: 0.986013986013986, 29: 0.7353324641460235, 30: 0.7888111888111888, 31: 0.8506787330316742, 32: 0.5164835164835165, 33: 0.4665012406947891, 34: 1.7353846153846153, 35: 1.0581613508442778, 36: 1.8076923076923077, 37: 1.972027972027972, 38: 0.7112232030264817, 39: 0.7611336032388664, 40: 1.1124260355029585, 41: 0.67788

In [ ]:
# Step 6: Model & augmentation

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.12),
    layers.RandomZoom(0.08),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomContrast(0.08),
])

base_model = EfficientNetB0(include_top=False, weights="imagenet",
                            input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling='avg')
base_model.trainable = False

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
x = layers.Lambda(preprocess_input)(x)
x = base_model(x, training=False)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = models.Model(inputs, outputs, name="effnetb0_transfer")
model.summary()

Model: "effnetb0_transfer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_16 (InputLayer)     │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_4 (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_4 (Lambda)               │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 1280)           │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 52)             │        13,364 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,397,015 (16.77 MB)

 Trainable params: 344,372 (1.31 MB)

 Non-trainable params: 4,052,643 (15.46 MB)

In [ ]:
cb_checkpoint = callbacks.ModelCheckpoint(
    MODEL_CHECKPOINT_PATH,
    monitor="val_accuracy",
    save_best_only=True,
    save_weights_only=True,   # fine, now filename matches requirement
    mode="max",
    verbose=1
)

In [ ]:
# Step 7: Compile & callbacks

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=BASE_LR),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=['accuracy']
)

cb_checkpoint = callbacks.ModelCheckpoint(MODEL_CHECKPOINT_PATH, monitor="val_accuracy",
                                          save_best_only=True, save_weights_only=True, mode="max", verbose=1)
cb_reduce = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=PATIENCE_RLR, verbose=1)
cb_early = callbacks.EarlyStopping(monitor='val_accuracy', patience=PATIENCE_ES, restore_best_weights=False, verbose=1)


In [ ]:
# Step 8: Stage 1 training

history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=INITIAL_EPOCHS,
    class_weight=class_weights,
    callbacks=[cb_checkpoint, cb_reduce, cb_early]
)


Epoch 1/8
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.2718 - loss: 3.2658
Epoch 1: val_accuracy improved from -inf to 0.76950, saving model to best_effnetb0.weights.h5
71/71 ━━━━━━━━━━━━━━━━━━━━ 275s 3s/step - accuracy: 0.2739 - loss: 3.2571 - val_accuracy: 0.7695 - val_loss: 2.1577 - learning_rate: 0.0010
Epoch 2/8
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7881 - loss: 1.3709
Epoch 2: val_accuracy improved from 0.76950 to 0.86348, saving model to best_effnetb0.weights.h5
71/71 ━━━━━━━━━━━━━━━━━━━━ 255s 3s/step - accuracy: 0.7882 - loss: 1.3711 - val_accuracy: 0.8635 - val_loss: 1.4954 - learning_rate: 0.0010
Epoch 3/8
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.8569 - loss: 1.1854
Epoch 3: val_accuracy improved from 0.86348 to 0.91135, saving model to best_effnetb0.weights.h5
71/71 ━━━━━━━━━━━━━━━━━━━━ 266s 3s/step - accuracy: 0.8568 - loss: 1.1858 - val_accuracy: 0.9113 - val_loss: 1.1864 - learning_rate: 0.0010
Epoch 4/8
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/ste

In [ ]:
# Step 9: Fine-tune top of base

base_model.trainable = True
for layer in base_model.layers[:-FINE_TUNE_UNFREEZE_LAST_N]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=FT_LR),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=['accuracy']
)

initial_epoch = len(history_stage1.history['loss'])
history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=initial_epoch + FINE_TUNE_EPOCHS,
    initial_epoch=initial_epoch,
    class_weight=class_weights,
    callbacks=[cb_checkpoint, cb_reduce, cb_early]
)

model.load_weights(MODEL_CHECKPOINT_PATH)

Epoch 9/33
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.8366 - loss: 1.2586
Epoch 9: val_accuracy did not improve from 0.95567
71/71 ━━━━━━━━━━━━━━━━━━━━ 323s 4s/step - accuracy: 0.8367 - loss: 1.2588 - val_accuracy: 0.9539 - val_loss: 1.0402 - learning_rate: 1.0000e-05
Epoch 10/33
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.8495 - loss: 1.2332
Epoch 10: val_accuracy did not improve from 0.95567
71/71 ━━━━━━━━━━━━━━━━━━━━ 312s 4s/step - accuracy: 0.8495 - loss: 1.2335 - val_accuracy: 0.9433 - val_loss: 1.0590 - learning_rate: 1.0000e-05
Epoch 11/33
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.8735 - loss: 1.1698
Epoch 11: val_accuracy did not improve from 0.95567
71/71 ━━━━━━━━━━━━━━━━━━━━ 323s 5s/step - accuracy: 0.8735 - loss: 1.1702 - val_accuracy: 0.9433 - val_loss: 1.0611 - learning_rate: 1.0000e-05
Epoch 12/33
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.8905 - loss: 1.1308
Epoch 12: val_accuracy did not improve from 0.95567

Epoch 12: ReduceLROnPlate

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 90 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
# Step 10: Evaluation

# Make sure val_ds contains only the 4 classes used in training
# Example: you have already filtered val_ds to contain only the relevant folders

print("\nEvaluating on validation set...")

y_true = []
y_pred = []

for x_batch, y_batch in val_ds:
    # Convert one-hot labels to class indices
    y_true_batch = np.argmax(y_batch.numpy(), axis=1)
    y_true.extend(y_true_batch)

    # Model predictions
    preds = model.predict(x_batch, verbose=0)
    y_pred_batch = np.argmax(preds, axis=1)
    y_pred.extend(y_pred_batch)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

val_accuracy = np.mean(y_true == y_pred)
print(f"Validation Accuracy: {val_accuracy:.4f}")

print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=class_names))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))



Evaluating on validation set...
Validation Accuracy: 0.0889

Classification report:
              precision    recall  f1-score   support

    Cattleya       0.08      0.08      0.08        13
  Dendrobium       0.12      0.15      0.13        13
Phalaenopsis       0.00      0.00      0.00         6
    Oncidium       0.07      0.08      0.07        13

    accuracy                           0.09        45
   macro avg       0.07      0.08      0.07        45
weighted avg       0.08      0.09      0.08        45

Confusion matrix:
 [[1 7 1 4]
 [2 2 0 9]
 [4 2 0 0]
 [6 6 0 1]]


In [ ]:
# Step 11: Optional Test-Time Augmentation (TTA)

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications.efficientnet import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Make sure augmentation is defined here
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.12),
    layers.RandomZoom(0.08),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomContrast(0.08),
])

USE_TTA = True  # Enable or disable TTA

def tta_predict(model, dataset, tta_steps=4):
    all_preds = []
    for _ in range(tta_steps):
        preds_step = []
        for x_batch, _ in dataset:
            x_aug = data_augmentation(x_batch, training=True)
            x_proc = preprocess_input(x_aug)
            preds_batch = model.predict(x_proc, verbose=0)
            preds_step.append(preds_batch)
        preds_step = np.vstack(preds_step)
        all_preds.append(preds_step)
    return np.mean(all_preds, axis=0)

if USE_TTA:
    print("\nRunning Test-Time Augmentation (TTA)...")

    # Ensure y_true exists (from previous evaluation)
    if 'y_true' not in globals():
        y_true = np.concatenate([np.argmax(y.numpy(), axis=1) for x, y in val_ds], axis=0)

    # Run TTA
    tta_preds = tta_predict(model, val_ds, tta_steps=3)
    tta_labels = np.argmax(tta_preds, axis=1)

    print("\nClassification report (TTA):")
    print(classification_report(y_true, tta_labels, target_names=class_names))
    print("TTA Confusion matrix:\n", confusion_matrix(y_true, tta_labels))
else:
    print("TTA is disabled. Set USE_TTA = True to run it.")



Running Test-Time Augmentation (TTA)...

Classification report (TTA):
              precision    recall  f1-score   support

    Cattleya       0.27      0.31      0.29        13
  Dendrobium       0.29      0.31      0.30        13
Phalaenopsis       0.00      0.00      0.00         6
    Oncidium       0.27      0.31      0.29        13

    accuracy                           0.27        45
   macro avg       0.20      0.23      0.22        45
weighted avg       0.24      0.27      0.25        45

TTA Confusion matrix:
 [[4 4 1 4]
 [2 4 0 7]
 [3 3 0 0]
 [6 3 0 4]]


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

y_true = []
y_pred = []

for x_batch, y_batch in val_ds:
    y_true_batch = np.argmax(y_batch.numpy(), axis=1)
    y_true.extend(y_true_batch)

    preds = model.predict(x_batch, verbose=0)
    y_pred_batch = np.argmax(preds, axis=1)
    y_pred.extend(y_pred_batch)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

val_acc = np.mean(y_true == y_pred)
print(f"Validation Accuracy: {val_acc:.4f}")

# Optional: detailed report
from sklearn.metrics import classification_report, confusion_matrix
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))
print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))


Validation Accuracy: 0.0889

Classification Report:
              precision    recall  f1-score   support

    Cattleya       0.08      0.08      0.08        13
  Dendrobium       0.12      0.15      0.13        13
Phalaenopsis       0.00      0.00      0.00         6
    Oncidium       0.07      0.08      0.07        13

    accuracy                           0.09        45
   macro avg       0.07      0.08      0.07        45
weighted avg       0.08      0.09      0.08        45


Confusion Matrix:
[[1 7 1 4]
 [2 2 0 9]
 [4 2 0 0]
 [6 6 0 1]]
